# Gradually Varied Flow (GVF) Analysis
## Water Surface Profile in a Rectangular Open Channel

This notebook computes and plots the **water surface profile** of gradually
varied flow (GVF) in a rectangular open channel using the **Direct Step Method**.

### Theory
In GVF the flow depth changes gradually along the channel.  The governing
equation is:

$$\frac{dy}{dx} = \frac{S_o - S_f}{1 - Fr^2}$$

where **Sₒ** is the bed slope, **Sf** is the friction slope (from Manning's
equation), and **Fr** is the Froude number.

The **Direct Step Method** integrates this by computing **Δx** from the change
in specific energy:

$$\Delta x = \frac{\Delta E}{S_o - \bar{S}_f}$$

### Profile Classification
| Profile | Condition |
|---|---|
| M1 | Mild slope, y > yₙ > yc |
| M2 | Mild slope, yₙ > y > yc |
| M3 | Mild slope, yₙ > yc > y |
| S1 | Steep slope, y > yc > yₙ |
| S2 | Steep slope, yc > y > yₙ |
| S3 | Steep slope, yc > yₙ > y |

### How to use
1. Edit `data/hydrau.xlsx` with your measured channel data.
2. Set `EXCEL_PATH`, volume, Manning's n, and channel width below.
3. Run all cells (`Runtime → Run all` in Colab, or `Kernel → Restart & Run All` locally).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

# Display settings
pd.set_option("display.float_format", "{:.5f}".format)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})


## 1. Load Data & User Inputs

In [ ]:
# ── USER: update these values ─────────────────────────────────────────────────
EXCEL_PATH = "data/hydrau.xlsx"   # path to the Excel input file

vol = float(input("Enter volume in litres: ")) / 1000   # convert to m³
man = float(input("Manning's roughness coefficient (n): "))
B   = float(input("Channel width (m): "))
# ─────────────────────────────────────────────────────────────────────────────

df  = pd.read_excel(EXCEL_PATH)
sl  = len(df)   # number of cross-sections

print(f"\nLoaded {sl} cross-sections from '{EXCEL_PATH}'")
print(f"  Volume      : {vol*1000:.2f} L  ({vol:.5f} m³)")
print(f"  Manning's n : {man}")
print(f"  Width (B)   : {B} m")
print()
print(df.to_string(index=True))


## 2. Hydraulic Calculations

All intermediate hydraulic quantities are computed column-by-column:
- **Discharge Q** = Volume / Time
- **Flow Area A** = B × y
- **Wetted Perimeter P** = B + 2y
- **Hydraulic Radius R** = A / P
- **Velocity V** = Q / A
- **Velocity Head** = V² / 2g
- **Specific Energy E** = y + V²/2g
- **Friction Slope Sf** = (Vn / R^⅔)²
- **Δx** = ΔE / (Sₒ − S̄f)


In [ ]:
# Discharge at each section
df["discharge"]          = vol / df["time:"]

# Cross-section geometry
df["Area"]               = B * df["depth_'y'(m)"]
df["Wetted_perimeter"]   = B + 2 * df["depth_'y'(m)"]
df["hydraulic_radius_R"] = df["Area"] / df["Wetted_perimeter"]

# Manning's velocity components
df["R^2/3"]              = np.power(df["hydraulic_radius_R"], 2 / 3)

# Flow velocity and energy
df["velocity"]           = df["discharge"] / df["Area"]
df["velocity_head"]      = df["velocity"] ** 2 / (2 * 9.81)
df["Energy"]             = df["depth_'y'(m)"] + df["velocity_head"]

# Change in specific energy between sections
df["delta_Energy"]       = df["Energy"].diff()
df.loc[0, "delta_Energy"] = df["Energy"].iloc[0]   # first section: full energy

# Friction slope Sf = (V * n / R^(2/3))^2
df["Sf"] = np.power(df["velocity"] * man / df["R^2/3"], 2)

# Average friction slope between consecutive sections (trapezoidal rule)
df["Sf_avg"] = df["Sf"].copy()
for i in range(1, sl):
    df.loc[i, "Sf_avg"] = (df["Sf"].iloc[i] + df["Sf"].iloc[i - 1]) / 2
df.loc[0, "Sf_avg"] = df["Sf"].iloc[0]

# So - Sf_avg
df["So-Sf_avg"] = df["Bed_slope"] - df["Sf_avg"]

# Step length Δx = ΔE / (So - Sf_avg)
df["delta_x"] = df["delta_Energy"] / df["So-Sf_avg"]
df["delta_x"] = df["delta_x"].replace([np.inf, -np.inf], np.nan).fillna(0)
df.loc[0, "delta_x"] = np.nan   # undefined at first section

# Cumulative distance x
df["Calculated_x"] = 0.0
for i in range(1, sl):
    df.loc[i, "Calculated_x"] = df.loc[i - 1, "Calculated_x"] + df.loc[i, "delta_x"]

print("Hydraulic calculations complete.")
print()
print(df[[
    "depth_'y'(m)", "discharge", "velocity", "velocity_head",
    "Energy", "Sf", "delta_x", "Calculated_x"
]].to_string())


## 3. Critical Depth (yc) and Normal Depth (yn)

- **yc** is where the Froude number = 1.  For a rectangular channel: `yc = (q²/g)^(1/3)`
- **yn** is found by solving Manning's equation numerically.


In [ ]:
g          = 9.81
Q_total    = vol / df["time:"].mean()          # average total discharge
q_unit     = Q_total / B                       # unit discharge (per unit width)
So         = float(df["Bed_slope"].iloc[0])    # bed slope (assumed uniform)

# Critical depth
yc = (q_unit ** 2 / g) ** (1 / 3)

# Normal depth via Manning's equation (numerical root-finding)
def manning_residual(y):
    A = B * y
    P = B + 2 * y
    R = A / P
    return (1 / man) * A * (R ** (2 / 3)) * (So ** 0.5) - Q_total

yn = fsolve(manning_residual, x0=yc)[0]

print(f"Total discharge Q    : {Q_total:.5f} m³/s")
print(f"Unit discharge q     : {q_unit:.5f} m²/s")
print(f"Critical depth  (yc) : {yc:.4f} m")
print(f"Normal depth    (yn) : {yn:.4f} m")

# Profile classification
slope_label = "M" if yn > yc else "S"
y_first     = df["depth_'y'(m)"].iloc[0]
if y_first > max(yn, yc):
    zone = "1"
elif min(yn, yc) < y_first < max(yn, yc):
    zone = "2"
else:
    zone = "3"
profile_label = f"{slope_label}{zone}"
print(f"\nProfile classification : {profile_label}")


## 4. Summary Table

In [ ]:
summary_cols = [
    "distance_'x'(m)", "depth_'y'(m)", "velocity",
    "Energy", "Sf", "Calculated_x"
]
# Use Calculated_x as the computed distance; keep original measured x for reference
print("=== GVF Summary ===")
print(df[summary_cols].to_string())


## 5. Water Surface Profile

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

# Water surface
ax.plot(df["Calculated_x"], df["depth_'y'(m)"],
        marker="o", linestyle="-", color="royalblue",
        linewidth=2, label="Water Surface (measured y)")

# Normal depth line
ax.axhline(yn, color="green",  linestyle="--", linewidth=1.5,
           label=f"Normal Depth  yₙ = {yn:.3f} m")

# Critical depth line
ax.axhline(yc, color="red",    linestyle="-.", linewidth=1.5,
           label=f"Critical Depth yc = {yc:.3f} m")

# Annotations
ax.annotate(f"yₙ = {yn:.3f} m", xy=(df["Calculated_x"].max(), yn),
            xytext=(5, 4), textcoords="offset points",
            color="green", fontsize=9)
ax.annotate(f"yc = {yc:.3f} m", xy=(df["Calculated_x"].max(), yc),
            xytext=(5, 4), textcoords="offset points",
            color="red", fontsize=9)

ax.set_title(f"Water Surface Profile — {profile_label} Type", fontsize=13, fontweight="bold")
ax.set_xlabel("Calculated Distance x (m)", fontsize=12)
ax.set_ylabel("Water Depth y (m)", fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("water_surface_profile.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → water_surface_profile.png")


## 6. Specific Energy Profile

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(df["Calculated_x"], df["Energy"],
        marker="s", linestyle="-", color="darkorange",
        linewidth=2, label="Specific Energy E (m)")
ax.plot(df["Calculated_x"], df["depth_'y'(m)"],
        marker="o", linestyle="--", color="royalblue",
        linewidth=1.5, alpha=0.7, label="Depth y (m)")

ax.set_title("Specific Energy along the Channel", fontsize=13, fontweight="bold")
ax.set_xlabel("Calculated Distance x (m)", fontsize=12)
ax.set_ylabel("Energy / Depth (m)", fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("energy_profile.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → energy_profile.png")


## 7. Friction Slope vs Distance

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(df["Calculated_x"], df["Sf"],
        marker="^", linestyle="-", color="crimson",
        linewidth=2, label="Friction Slope Sf")
ax.axhline(So, color="gray", linestyle=":", linewidth=1.5,
           label=f"Bed Slope Sₒ = {So}")

ax.set_title("Friction Slope along the Channel", fontsize=13, fontweight="bold")
ax.set_xlabel("Calculated Distance x (m)", fontsize=12)
ax.set_ylabel("Slope", fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("friction_slope.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → friction_slope.png")
